# KEV-4B on Colab A100 — Adaptive LLM Router Lab

**목적:** NVIDIA A100에서 공식 **Kev-4B**를 GPU 실행하고, `L1/L2/L3` 모델 선택 실험과 공식 KEV 벤치마크를 분리해서 수행합니다.

- [프로젝트 저장소](https://github.com/namyikim/adaptive-llm-router)
- [Kev 공식 저장소](https://github.com/jaredpalmer/kev)
- [Kev-4B 모델](https://huggingface.co/jaredpalmer/kev-4b)

> **주의:** 샘플 데이터의 `l1/l2/l3` 라벨은 설명용으로 부여한 값입니다. 모델별 **실제 정답률은 측정하지 않습니다**. 이 실험만으로 비용 절감이나 최적 라우팅을 입증할 수 없습니다.

**Colab 메뉴: `런타임 → 런타임 유형 변경 → A100 GPU`**. 별도 GitHub PAT 없이 공개 저장소를 clone할 수 있습니다.

**독립 실행 지원:** 프로젝트 GitHub 저장소가 아직 비어 있다면 노트북에 포함된 경량 라우팅 예제와 테스트 데이터를 자동으로 복구합니다. 단, 공식 Kev 소스와 모델은 Colab에서 인터넷으로 다운로드해야 합니다.

## 1 · GPU 확인

In [ ]:
import os, shutil, subprocess, sys, json, time
from pathlib import Path

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError("NVIDIA GPU를 찾을 수 없습니다. Colab 런타임을 GPU(A100)로 변경하세요.")
print("GPU 장치 확인 완료")

## 2 · 프로젝트 및 공식 Kev 소스 복제

Colab 인스턴스의 `/content`에 복제합니다. `pip install kev`는 **다른 PyPI 패키지**이므로 사용하지 않습니다.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil, os, json, zlib, base64
ROOT = Path("/content/adaptive-llm-router")
UPSTREAM = Path("/content/kev")
OUTPUTS = Path("/content/outputs")
OUTPUTS.mkdir(parents=True, exist_ok=True)
if not (ROOT / "src/adaptive_llm_router/routing.py").exists():
    clone = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/namyikim/adaptive-llm-router.git", str(ROOT)], capture_output=True, text=True)
    if clone.returncode != 0 or not (ROOT / "src/adaptive_llm_router/routing.py").exists():
        print("GitHub project files are not published yet; using the self-contained notebook demo fallback.")
        embedded = json.loads(zlib.decompress(base64.b64decode('eNqdWmt3G7cR/Ssov4hUSOqVtAkdpnUdtfWJYyuSnHwQddbgLkgi2lcWu5QZlv+9dwbYJylFaXxOqMUCg5nBzJ0HdtszmX8iA5nmeq28MIy8LClylZ14no517nnjdNObiN6M//376uNoLv0HFYgf1FoEytdGJ7GIkkCFQse5WmYypxEZB8KXxpeBEupzqjIdqTjnd2NLaxbPYs9bq4woeJ6YilnvdHw2PqVXvaF4krMHtfb8UINci7dbHW+EybGxzIJRqOeZzDbCThSLJCOOj4y43aTqRi7UyE+iFPzMQyVeX71tMLXIkkh43qLIi0yBMR2lSZZDojixAhia5UZ/NUm5pMhC7DpWWYbN3Ov/3N5eXdLAUHy8fsd/tWdn6rdCmbycf20fh/Q6SVU8439+KI0h/t8kcax84oFJ9a+LOIdi+WEwmcUC/1lBrqU2OKXHlYpFvlIiTHwZCpxffWhGZdC+0EZAMCHXUocS2mhoYhYHaiHSxOSe2ZhcRUms+nYXKDpXE/xk4r8i0H4+tOPMPulo0hw9dr9zaZQH0exCOvFVnqeTk5Oz87+NT/HvbPL16ek3s56bT8Lh0CdiESYyx4Kzc8xyL2WqvQe1KZl4D+Ywg34wYSBG3zEHTiup3IBEgAnbWY/NdQbTmfVgTKMQopgcm+KZ5aJX/AeNVBLRaPWws2RXCgaeGUsWp5PD2EZkYpa6TNNQ+2w0J2wpPbdOLyr27UCD2N2s97rIV0mmf+eVs9496C9mvX8qmeHAtm7ljg6J1pU2NC3Np1/TLDU+zqAlnfZnvZNZbyC+AHMn67OT6lgrldN/gczllPgdB0WUmr5T3lCo2JBPwK+1nv5LhkYNxir2oU4QLvLF6GsQH+5JNHW/jTeRgoTBdNa7+nBzW20+cMeeNdXyqPNV6Q/9rHQQZxpT9zsQ0kARJsXZqMbiUhwohwUiOfrlPLed+uyrtOGrRApjDSom1mmqSMMYh8+Cxlenp4MxfKklOzREBAzkylQaSh96HdRkMnLKQ36MwyU8JQ7ElrYgsruJ2LqNd3RmDBt42WK6X8LKUNxaTbinDzcWFfaF+SMuHC4UMeT0VwQJE2bqABMwY8IO4ElMyOurPul6yI43ENDkgbfjpYKB9mRsHmESZC92+h6HP8uwsNCG6R9jiiJ+7iJPeYKiH2kD+kvh6A0qhWcK+B3z4T8fT+gHFFrB5NqOieM0S4B/Mjy2kmPvIcc2o1KJUKcEVKUXWtXRLk3g8psxoSdxmmaKxCPMEHBf1siywFpgBVblCVxBkleJd+9+dFIIP8kyCBsrY8bitVgVkYwFhQ78aIAyPCGgsJZL8zAy0AtY8EtWMhEVQANwvtbgh9A/Rfw4WQDgK/Q3xOCfDHk8i/TJ8UiZclo1VE0a1zG6nNSOIjTz9u3l9Q1cCqcbnln0Dc/d7wUf4/WHj7eX3k8fL29u3354zyDrQhzsKcu9HMIS0G5r05n18gp9/VWi/TayzXpki1nhV4jeb2PFrHejQqjeBk2cTI6DTckJBO0mQv2gwg0dm0nCNWkX8bOAy5QoPBYlLNck36ySBLYannEeYqCRkLKiPJPMB1BDGkpVoEI6SJtBYSYmxCbkx1cHyIbnTI+iGZsiThzpVxTBcIdgRwLvyIZpy00MeYw2B8lcMBnKh0L1GcYT5hrHlDZIkMXLYE0+HMCE5sVySR5zgNj3ic0mjCkia3v2FNgcYTISSRqHX6RiTStvUhq0DszPNLxUy85BO97P7FHfRDKkBJRAIjZwcPamWttDShKkXq5yjD0iQWTXMS3bKJVqKf4Ity2iigzltlmEMVJ0pZhhU+cWF0pNHyJ94ZjNswTTa/WWmwR6geOH/huHsM/nzv2NVGJns7R/VB7Yh/v9ruLpbVZQdLOJI4FZGRPJiDljcmkRw5sKvO64F8nPLu2qZs7lXIcaM11yd4fpQzvn3k4qYCBZLnU8EfMkCe0gpVexv/EiU+ZxNllrZJgyQ5bCcFzFZpc/InUUUK5Xb7+ps8HT8V+/Gtab1kJwakknyWlgU34XsZBFim+nXco0hALk+UjUXcNgO1ecpuhY3J0Oxdl9FYWwX5s/GxJjwej33E6Ixm/jtQx1sCfhtj2wq3Zz4WNaxce7RqC9v2vj5n2divIUF5gtfg7EX6YNCH1eJZdlaHa+bulVXJG2KEVubdOyJ0L97a5WmcEUXsVs0BOr6+U5gi1IWd/Ns0LWbPbY2j44i7Jb3j3cD9gbH6pjaiTt8aa/Ft+KU0LVtfhOWEhf01RePV4TR6Y/qNKf/un4m6/JsoCI/e4ca3DnfyDYjzLELlElV0d3g7YvQyQ4r9XYkESe2l09D5oHlkaeN2h4OaZbucv1XV+mCTzx266/1JCCOR0zbxq+UKgUKv5ceiYfPYcMDXt1xlFDRjedY1/u0w7DiuDQ8je0cjQAwYFTv7HXgBhr7u3KX64emU36a1BjE6OS54J7vy57GZleXs8+D2J7MFXlt9Nu+e3q0nZqVHMyLf9oq+0Qwu7xNO08N7QAz7Kw0fcLBO04r3WANyqLEX43XpkLe5Rv2hDAktUFertHYXlj1eNUFjrWZkXHSbuJ0ooobaVBuBPPhNLeXQhIsdbqEVNVoIIBp9tE+bX4xHXpJzcPx3t8/P7D7fEx/M/3OaG+pawEDJeZMrLvBUo1wWBrky/aFKnwuMluBQJOBX8GyT/GD3HyGLvwu3UUds0w8aQeG5RrhbllAbkvMzCmuuBzeTzt0+cJdzT5C8Qm2sz9DZcOUVRbcK3Nvz53V9T01WflF2R3zNqG0qkykLAldNstVmOXdpWgY+PjxJlIrtyHnC7rzKZN4u377y+vLvG/97d1FQM7RRZUnewnxwM7/+CTk81wjmkjyyvxybLn8MGONqZSVgRmQgdccgGNI52lzI5sL0cN5a+U/8CZ3RLeAh0EQhFcVzm5rcWs9R02j1o1/0+s59WlndSkGjE+BxKkOUWuOwfVFIKsu5Rn3TSJmshgct/gokoWWoqt35PxcTQ5pNR62n4PwC4cWv/v9GKs6NQhK4Pbz62q1Vms5MVKxq3+SSn6WKZUAfe3yFZcCWiZm/VK16Exy8euzasd7HJld93a/l9hbJ5e0yLK3Z3KJGfiVMJjjkEedX/vWq7Y2cLakWedobsTtzJbO1Ujh/d5rsvhrFi12hyXbpB7KsehnCO5kMtMKWrWH8NVhA7DgkonIgcZULIFRvQBqKX5/FZIjhX/bwNdZksOToc66n4Shsr1TtzbN3BKiOMmpDJfhXpevrzCY92DcH2d8qXLh1rxvBHmlQw8oJ2ibme+KpvKRJGxLQQ83BHAlR6UJY9NF+QmJXcoaTl1Sv0kwO7Tui9KnbhFw/DIa5GihJrCH3KkGBUz1fH9hYPX6VnXe5yv0ZKx7ed2Z9B/fhJD8EJ1bZwOr9kHNX0iNDi4RcOd7UKXmbn++GBoUT/JROs1mxDJ+gTyvRABKWZkbdtzxmmFF9t410KG8kRKZLBMDVo1H71/Ps/+HmU0ig1KGsilNt3kkyjUJoN8PpKZ/h0aIsoNE+lGwxcz8D5xUc9wh6ncoOIjhwOFOEIK2kTNDbtGirA1RnZH1VXZdeTbAyTYd9Xh3LPlcbBo0sBq51z9fnP2UOwRHBykUCFco3kTM2IS1+0mXONkPd7Iq2CHVpQCnRxYW9cZC9c/4+saCE4hg1mvphzg9SBNeMyiMI7UdpmEAX5IZLDClt2nsSE3c2H6p5YoPdU1Ig/RtHqotQcswPUlX5eiWtQwxfxXQjkCWAdZrBLzyrbS/BwQyz0il8EK7tZxsAxsJ3NcNYh2tX1GUECfTbGReadUhzvIHb/OlgXxcUVPWb+sCscyCDzp3sEqRyMdpwVfkJWBqmwwPTEfYvy5BVS1jFC10BLwLuEC0yeLqafJII0bNetTUKN2xpRrrZoyl1wum8qWBOPp2FZI9FjqoYPw3AUcljDaAuxG8CACY1YX7KSF32wt0049aandVbh636jkmFT5tF+s8ev9iu0QGu61TTXnRy3k1px86AG3363rlzOaYNCl1MSFCcs4LhOkqt4vX7SajQcotZoa9Zp6lKlSqV+/5sK/S6rh/25eoyXgrn+9Zo/BzaqHmu3WZkqMgiWnWLXVk+8udmLbUOGRDo5IgbuJdd+pe3l3xI9H97vyLgjesK00hVGI4AZs23V8sdg1Qg9nL3zBbAPCxuFdI/jwqbnsjM8NYy7vtK6I9ZTKWPO0Q4PmezJ/+qAiegh01rcPhn2WCnwENi95aLqwW/VIvXkvR+nab1wXW5aHfFMV59Pzg/fGfAU9o0uog8lS2dJy+v4lA3qKrd1290o8F0BI8cTA3ZHT19H93dFzC47ua5VzZrpA9hrLiHLXKTV1PI+w1POqPqiFVptwz8H9Cts8mBNqw58EKkq88k6RtBJSvr11TodD5AuJ0SldedXfG8x6t+6yR4mjZZJQqy+jy4EjygR+SAAxFuUr53S3ZgheWLNM2CxQO9RXRvyxwf6+5519L+09FNfRsB+qWotojh/KoifiAw+dnV98+ZUwK7qNpkvLQG5ewk19x/UEMxcdZn5ZSU6/vhEpzkt8+fcXbAI3yFeRQvX+xCZfdjZ5Yy/aNvbyzjWSkJwDnDSHYUrM1ZJtZSLeijBZk9DQD12Na1+9RPT2bV6Hs0gH+wbwCzkTKt+rTb6iz3N85Pe5vSDOlnSXBnMVb25+FgsdKmNzB7oxUtyMQq1URLEgswVOulYWOPZXKpJ8NRUmdOf0q22Qc3joinF+QAx2zEPs79tRGlKv9+and+LXBNWDKPhiXn2WfN+Z0yWqYU4CbfwCeYz9Oimh+1NB3xrwOo7YL2Gtuj87xF3XsK7tKfOJX1/e3NI3Xygflzq2nZ1AGb20n625MCVWeAhJAor86nNKaQz1/kySWTGQ8GaISi/h1ZI/xGjXOG9KYKdOmfJXsaYPtyKlODFk97aGAWRM6uv8oUgeY+UYozq0/lTpJfyV4eSQsa7oe7o9a0W1RvSXkm0WJwpdzIuc74dlECb+A90oz7W9Pzab2NeJ8KmgDC08DcUVclhgMBkMzJE+UXsw9usKmY/47n2UYD6EN0YuVanvV9UJ0b5yGcNtDd/vQqB9WLo4dBruBvugpF3D/t6ZhuDkcZQnoaKvN+wd7ShTS7KfVG44ow9VsAResnPC8v083Fgh5uQNlLwreKyfk6BFoPPyZsq5BWRWtqReq2xDxx2oZLEwL5FKZv4KCOJT1+WgYF2fuKIrecgVJzEUu9aSbtDXMAPpvpiUsT25VZbESUEX9rGhUG7oBPIEkGMFRb6Q0+pyrhVHOUBwvXT7TUB62CQPiEM3XjIcPeXlVqSu93wA/Yidh/LzAIc1wg9/96hjRkq/QnFk2iHskezzQWWxCoWtAYfU04pz5wz2iy7UDhtAq/axKIea3W1/Tp/UlDdJ1C3OYKwI3BtK9YzhG4Q/ltRe7hgWsLf7HztFp7U=')).decode("utf-8"))
        for relative, source in embedded.items():
            destination = ROOT / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_text(source, encoding="utf-8")
if not (UPSTREAM / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jaredpalmer/kev.git", str(UPSTREAM)], check=True)
print("Project:", ROOT, "Kev:", UPSTREAM)


## 3 · Kev Python 3.13 환경 설치

공식 `uv sync --extra serve`로 전용 가상환경을 설치합니다. GPU 커널 및 모델 다운로드는 Colab 환경 상태에 따라 시간이 달라집니다.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
assert uv, "uv 설치 확인 실패"
subprocess.run([uv, "sync", "--extra", "serve"], cwd=UPSTREAM, check=True)
p = subprocess.run(
    [uv, "run", "--extra", "serve", "python", "-c",
     "import sys, torch; print('Python:',sys.version); print('torch:',torch.__version__); print('CUDA available:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); assert torch.cuda.is_available()"],
    cwd=UPSTREAM, text=True, capture_output=True
)
print(p.stdout or p.stderr)
p.check_returncode()

## 4 · Kev-4B 서버 시작

로컬 루프백 `127.0.0.1:8009`로만 서비스합니다. `@v1.0`은 공식 KEV 모델 가중치 태그입니다. 첫 실행에서는 Hugging Face로부터 모델 가중치를 다운로드합니다.

In [ ]:
import subprocess, os, time
log_path = Path("/content/kev_server.log")
if "kev_server" in globals() and kev_server.poll() is None:
    print("Already running:", kev_server.pid)
else:
    log_file = log_path.open("a", encoding="utf-8")
    kev_server = subprocess.Popen(
        [uv, "run", "--extra", "serve", "python", "-m", "kev.serve",
         "--run", "jaredpalmer/kev-4b@v1.0", "--device", "cuda",
         "--host", "127.0.0.1", "--port", "8009"],
        cwd=UPSTREAM, stdout=log_file, stderr=subprocess.STDOUT
    )
    log_file.close()
    print("Started PID", kev_server.pid)
print("Log:", log_path)

## 5 · 서버 준비 확인

로딩 중에는 서버 로그 끝부분을 확인합니다. 반복 확인이 실패하면 `/content/kev_server.log`에 나타난 오류를 확인하세요.

In [ ]:
import urllib.request, urllib.error, time
ready = False
for attempt in range(120):  # at most 20 min including download
    if kev_server.poll() is not None:
        print(log_path.read_text(encoding="utf-8", errors="replace")[-4000:])
        raise RuntimeError(f"Kev process exited: {kev_server.returncode}")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8009/v1/models", timeout=5) as resp:
            info = json.load(resp)
            ready = True
            print("Kev server is ready", json.dumps(info, ensure_ascii=False)[:600])
            break
    except (urllib.error.URLError, TimeoutError):
        if attempt % 6 == 0:
            print(f"Still loading ({attempt * 10}s), see {log_path}")
        time.sleep(10)
if not ready:
    print(log_path.read_text(encoding="utf-8", errors="replace")[-4000:])
    raise TimeoutError("Kev didn't become ready: inspect server log")

## 6 · KEV가 L1/L2/L3를 고르는지 확인

KEV는 **텍스트를 생성하지 않고 선택 확률을 반환**합니다. 아래 질문은 학습된 전용 라우팅 기준이 아닌 **제로샷 휴리스틱**입니다.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from adaptive_llm_router.routing import route_request

examples = [
    "Extract invoice number 1024 from this message.",
    "Write a Python data-processing pipeline with schema validation and tests.",
    "Diagnose a multi-region distributed transaction race condition in a payment system."
]
for query in examples:
    result = route_request(query)
    print(json.dumps({
        "query": query, "proposed": result.proposed_tier, "chosen": result.tier,
        "p_max": result.p_max, "probabilities": result.probabilities,
        "uncertain": result.uncertain, "model_latency_ms": result.latency_ms,
    }, ensure_ascii=False, indent=2))

## 7 · 샘플 데이터셋 평가 (12개)

혼동행렬과 샘플 라벨 일치율을 저장합니다. **실제 LLM 성능/정답률 평가가 아닙니다.** 결과는 `/content/outputs/router_demo.json`에 생성됩니다.

In [ ]:
import os, subprocess, json
env = dict(os.environ, PYTHONPATH=str(ROOT / "src"))
subprocess.run([
    sys.executable, "-m", "adaptive_llm_router.evaluate",
    "--input", str(ROOT / "benchmarks/data/demo_routing.jsonl"),
    "--output", str(OUTPUTS / "router_demo.json"),
], env=env, check=True)
report = json.loads((OUTPUTS / "router_demo.json").read_text(encoding="utf-8"))
print(json.dumps(report["summary"], ensure_ascii=False, indent=2))

## 8 · 공식 KEV Benchmark (선택)

공식 `devtools-v1` 벤치마크의 **development split**을 Kev 자체 평가 절차로 실행합니다. 샘플 라우터 평가와 별개입니다. 데이터 파일이 필요하면 공식 Hugging Face 미러에서 다운로드합니다. 1,000개 내외의 질문을 평가하므로 추가 GPU 실행 시간이 필요합니다.

아래 `RUN_OFFICIAL_BENCHMARK = True`로 변경하여 실행하세요.

In [ ]:
RUN_OFFICIAL_BENCHMARK = False
if RUN_OFFICIAL_BENCHMARK:
    bench_out = OUTPUTS / "kev_devtools_benchmark"
    if bench_out.exists():
        raise FileExistsError(f"Output exists: {bench_out}; change or remove it before rerun")
    subprocess.run([
        uv, "run", "--extra", "serve", "python", "-m", "kev.benchmark",
        "--remote", "http://127.0.0.1:8009",
        "--suite", "evals/devtools-v1",
        "--split", "development",
        "--out", str(bench_out),
    ], cwd=UPSTREAM, check=True)
    official = json.loads((bench_out / "report.json").read_text(encoding="utf-8"))
    print("Official Kev devtools-v1 development benchmark:")
    print(json.dumps({
        "clean": official.get("clean"), "coverage": official.get("coverage"),
        "latency_ms": official.get("latency_ms")}, indent=2))
else:
    print("공식 benchmark 생략. 위 플래그를 True로 변경하면 실행합니다.")

## 9 · GPU 메모리 확인 / 실험 결과 다운로드

In [ ]:
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("Saved outputs:", *[str(p) for p in OUTPUTS.rglob("*") if p.is_file()], sep="\n  ")
archive = shutil.make_archive("/content/kev_gpu_results", "zip", OUTPUTS)
print("ZIP:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running inside Colab: download ZIP manually.")

## 10 · 서버 중지 (필요할 때 실행)

실험을 마친 후 GPU 메모리를 반환하려면 실행하세요.

In [ ]:
# Optional cleanup — uncomment after experiments.
# kev_server.terminate()
# kev_server.wait(timeout=30)
# print("Kev server stopped")